# 11 - ViT-S/16 - COVID-QU-Syn

This notebook is **repo-script-first**: it tries to use the repository's original scripts and modules instead of reimplementing training logic inside the notebook.

## Repo usage / self-coded inventory

**Goal**: ViT-S/16 run for strategy **COVID-QU-Syn**.

**Repo-original code used**

- `scripts/run_dino.py` for DINO pretraining when this strategy needs contrastive/self-supervised pretraining.
- `scripts/run_classification_vit.py` for supervised ViT fine-tuning.
- `scripts/evaluate_classification_vit.py` is attempted after fine-tuning if a checkpoint is found.

**Still self-coded in this notebook**

- Colab setup, Google Drive mounting, dependency installation.
- Writing generated YAML config files for this strategy.
- Finding the checkpoint path after pretraining/fine-tuning.
- Running the repo scripts in the correct order and saving logs.

**Not self-coded here**

- No custom DINO loop.
- No custom ViT fine-tuning loop.

In [ ]:
# ============================================================
# Colab setup: clone repo, mount Drive, install dependencies
# ============================================================
import os, sys, subprocess, json, shutil, textwrap, time
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Not running in Colab; continuing with local paths.')

REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_DIR = Path('/content/contrastive-synthesis-medcls_CVProject') if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])
    else:
        print('Repo already exists:', REPO_DIR)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=False)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('Repo dir:', REPO_DIR)

# Install dependencies. Keep this lightweight; repo requirements are attempted first.
req = REPO_DIR / 'requirements.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=False)
if req.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', 'torchaudio', 'timm', 'PyYAML', 'scikit-learn',
    'seaborn', 'matplotlib', 'pandas', 'numpy', 'pillow', 'opencv-python', 'tqdm', 'wandb'
], check=False)

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU, running on CPU')

# Disable W&B prompts by default in Colab runs.
os.environ['WANDB_MODE'] = 'disabled'

# Paths
DRIVE_ROOT = Path('/content/drive/MyDrive/medcls_cvproject') if IN_COLAB else (REPO_DIR / 'local_drive_outputs')
DATA_ROOT = REPO_DIR / 'data' / 'processed'
LABELLED_DATA = DATA_ROOT / 'labelled_4232'
UNLABELLED_REAL_DATA = DATA_ROOT / 'unlabelled_16934' / 'images'
SYNTHETIC_DCGAN_DATA = DRIVE_ROOT / 'data' / 'processed' / 'synthetic_dcgan'
SYNTHETIC_ACGAN_DATA = DRIVE_ROOT / 'data' / 'processed' / 'synthetic_acgan'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs_script_first'
CONFIG_OUT = REPO_DIR / 'configs' / 'generated_script_first'
CONFIG_OUT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('LABELLED_DATA:', LABELLED_DATA)
print('UNLABELLED_REAL_DATA:', UNLABELLED_REAL_DATA)
print('SYNTHETIC_DCGAN_DATA:', SYNTHETIC_DCGAN_DATA)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

# Change to 'smoke' for quick testing.
RUN_MODE = 'full'
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

In [ ]:
# ============================================================
# Helpers: write YAML and run repo scripts with logs
# ============================================================
import subprocess, sys, yaml, os, json, time
from pathlib import Path

def write_yaml(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(data, f, sort_keys=False)
    print('Wrote config:', path)
    print(yaml.safe_dump(data, sort_keys=False))
    return path

def run_and_log(cmd, log_path=None):
    cmd = [str(x) for x in cmd]
    print('Running command:')
    print(' '.join(cmd))
    if log_path is None:
        subprocess.run(cmd, check=True)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as f:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            f.write(line)
        ret = process.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)

def count_images(root):
    root = Path(root)
    if not root.exists():
        return 0
    exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    return sum(1 for p in root.rglob('*') if p.suffix.lower() in exts)

def find_checkpoint(output_dir):
    output_dir = Path(output_dir)
    candidates = [
        output_dir / 'best_model.pth',
        output_dir / 'best_finetune_model.pth',
        output_dir / 'checkpoint.pth',
        output_dir / 'model.pth',
        output_dir / 'classifier.pth',
    ]
    for p in candidates:
        if p.exists():
            return p
    all_ckpts = sorted(list(output_dir.rglob('*.pth')) + list(output_dir.rglob('*.pt')), key=lambda p: p.stat().st_mtime, reverse=True)
    return all_ckpts[0] if all_ckpts else None

In [ ]:
# ============================================================
# ViT-S/16 strategy: COVID-QU-Syn
# Script-first execution order:
# 1. Optional DINO pretraining via scripts.run_dino
# 2. ViT fine-tuning via scripts.run_classification_vit
# 3. Optional evaluation via scripts.evaluate_classification_vit
# ============================================================
EXP_NAME = '11_vit_s16_covid_qu_syn'
STRATEGY = 'COVID-QU-Syn'
EXP_DIR = OUTPUT_ROOT / 'classification' / EXP_NAME
EXP_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if RUN_MODE == 'smoke':
    dino_epochs = 1
    ft_epochs = 1
    batch_size = 8
    local_crops_number = 2
else:
    dino_epochs = 120
    ft_epochs = 50
    batch_size = 32
    local_crops_number = 8

assert LABELLED_DATA.exists(), f'Missing labelled data: {LABELLED_DATA}'
print('Labelled count:', count_images(LABELLED_DATA))

dino_ckpt = ''
if True:
    PRETRAIN_DATA = SYNTHETIC_DCGAN_DATA
    assert PRETRAIN_DATA.exists(), f'Missing pretraining data: {PRETRAIN_DATA}. For synthetic runs, run notebook 00 first.'
    print('DINO pretraining data:', PRETRAIN_DATA, 'images:', count_images(PRETRAIN_DATA))
    dino_out = EXP_DIR / 'dino_pretrain'
    dino_cfg_path = CONFIG_OUT / f'{EXP_NAME}_dino.yaml'
    dino_cfg = {
        'data_path': str(PRETRAIN_DATA),
        'output_dir': str(dino_out),
        'num_workers': 0,
        'arch': 'vit_small',
        'patch_size': 16,
        'out_dim': 8192,
        'norm_last_layer': True,
        'use_bn_in_head': False,
        'warmup_teacher_temp': 0.04,
        'teacher_temp': 0.04,
        'warmup_teacher_temp_epochs': 10,
        'student_temp': 0.1,
        'center_momentum': 0.9,
        'momentum_teacher': 0.996,
        'epochs': dino_epochs,
        'batch_size': batch_size,
        'learning_rate': 0.001,
        'min_lr': 0.00001,
        'weight_decay': 0.04,
        'weight_decay_end': 0.4,
        'clip_grad': 3.0,
        'freeze_last_layer': 1,
        'use_fp16': torch.cuda.is_available(),
        'warmup_epochs': 5,
        'local_crops_number': local_crops_number,
        'global_crops_scale': [0.4, 1.0],
        'local_crops_scale': [0.05, 0.4],
        'saveckp_freq': 20,
        'seed': 42,
        'use_wandb': False,
        'wandb_project': 'lung_radiology_dino_pretrain',
        'wandb_entity': None,
        'wandb_run_name': EXP_NAME + '_dino',
        # Included for documentation; current repo script may ignore this flag.
        'use_imagenet_init': False,
        'imagenet_first': False,
    }
    write_yaml(dino_cfg_path, dino_cfg)
    run_and_log([sys.executable, '-m', 'scripts.run_dino', '--config', str(dino_cfg_path)], EXP_DIR / 'run_dino.log')
    found = find_checkpoint(dino_out)
    assert found is not None, f'No DINO checkpoint found in {dino_out}'
    dino_ckpt = str(found)
    print('DINO checkpoint selected:', dino_ckpt)
else:
    print('No DINO pretraining for this strategy.')

ft_out = EXP_DIR / 'finetune'
ft_cfg_path = CONFIG_OUT / f'{EXP_NAME}_finetune.yaml'
ft_cfg = {
    'data_path': str(LABELLED_DATA),
    'output_dir': str(ft_out),
    'num_workers': 0,
    'img_size': 224,
    'num_classes': 4,
    'train_split': 0.8,
    'val_split': 0.1,
    'arch': 'vit_small',
    'patch_size': 16,
    'embed_dim': 384,
    'num_heads': 6,
    'pretrained_checkpoint_path': dino_ckpt,
    'freeze_backbone': 'False',
    'epochs': ft_epochs,
    'batch_size': batch_size,
    'learning_rate': 0.0001 if 'covid_qu_syn' == 'none' else 1e-5,
    'weight_decay': 0.01,
    'use_fp16': torch.cuda.is_available(),
    'seed': 42,
    'early_stop_patience': 10,
    'use_wandb': False,
    'wandb_project': 'lung_radiology_classification',
    'wandb_entity': None,
    'wandb_run_name': EXP_NAME + '_finetune',
    # Included for documentation; current repo script may ignore this flag.
    'use_imagenet_init': False,
    'imagenet_first': False,
    'strategy': STRATEGY,
}
write_yaml(ft_cfg_path, ft_cfg)
run_and_log([sys.executable, '-m', 'scripts.run_classification_vit', '--config', str(ft_cfg_path)], EXP_DIR / 'run_classification_vit.log')

ft_ckpt = find_checkpoint(ft_out)
print('Fine-tuned checkpoint found:', ft_ckpt)
if ft_ckpt is not None:
    eval_out = EXP_DIR / 'evaluation'
    eval_out.mkdir(parents=True, exist_ok=True)
    eval_cmd = [
        sys.executable, '-m', 'scripts.evaluate_classification_vit',
        '--config', str(ft_cfg_path),
        '--checkpoint_path', str(ft_ckpt),
    ]
    # Some versions of the repo evaluation script also accept output/data overrides; if not, this still uses the config.
    try:
        run_and_log(eval_cmd, EXP_DIR / 'evaluate_classification_vit.log')
    except Exception as e:
        print('Evaluation script failed or is incompatible with this repo version:', repr(e))
        print('Fine-tuning script output/logs are still saved in:', ft_out)

print('Finished:', EXP_NAME)
print('Output folder:', EXP_DIR)